In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## 11.3 Whisper Transcript

In [19]:
import subprocess
from pydub import AudioSegment
import math
import glob
import openai 

In [15]:
audio = '/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/files/test_voice.mp3'

def extract_audio_from_video(video_path, audio_path):
    # ffmpeg -i files/podcast.mp4 -vn files/audio.mp3
    command = ["ffmpeg", "-i", video_path, "-vn", audio_path]
    
    subprocess.run(command)
    

def cut_audio_in_chunks(autio_path, chunk_size, chunks_folder):
    track = AudioSegment.from_mp3(autio_path)
    chunk_len = chunk_size * 60 * 1000
    chunks = math.ceil(len(track) / chunk_len)
        
    for i in range(chunks):
        start_time = i * chunk_len
        end_time = (i+1) * chunk_len
        chunk = track[start_time:end_time]
        
        chunk.export(f"{chunks_folder}/chunk_{i}.mp3", format="mp3")

defaultPath = '/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang'

cut_audio_in_chunks(f"{defaultPath}/files/test_voice.mp3", 5, f"{defaultPath}/files/chunks")


In [1]:
## open ai방식 
# import openai

# transcript = openai.Audio.transcribe("whisper-1", open("/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/files/chunks/chunk_0.mp3", "rb"))
# transcript["text"]


# open source 
import whisper

model = whisper.load_model("base")
result = model.transcribe("/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/files/chunks/chunk_0.mp3")
print(result["text"])



/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/env/lib/python3.11/site-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 네. 네. 여보세요. 고객님 통화 괜찮으실까요? 네. 네. 전화 잘 나와주셔서요. 네. 아. 다르면이랑 구입 용도를 구입 용도는 이런 건 언제 적어요? 구입 용도? 네. 그런 거죠. 구입 용도인지 상환 용도인지 이런 거 정냉기 없나요? 계약서요. 계약선 상위는 따로 없고 대출거래 확인서라는 게 있기는 한데, 저희는 그냥 스마트로 보내드리여서 적구시프시면 오셔서 적으셔도 돼요. 아. 계약선 상환 확인 요청서라는 게 있거든요. 그래가지고 이게 사실 대환하는 현장에서는 서류가 이게 들어가긴 들어가야 돼요. 그러니까 지금 현재 누구 채무로 되어 있는지, 뭐죠? 원호는 뭔지 용도는 어떻게 맞았는지 알고 계신 것처럼. 네. 그 기존이에요. 우리 고객님은 그냥 원래 구입하셨을 때 용도만큼만 받으시는 거잖아요. 네. 더 추가를 안 받으시고. 네. 그래서 고 가목을 그대로 살려드릴 거예요. 구입 용도를 받았다고. 아 고 구입 용도가 되는구나. 사실 그 금액 안 적으셨잖아요. 네. 그래서 감정까지 때문에 그거 오실 때 적으시고 가셔도 돼요. 저희 스마트로 보내드릴까 했었는데 그때 말씀해 주시면 그때 휘어 놓고 그때 적록 할게요. 사실은 이걸 여쭤봐도 되는 건 주도를 잘 모르겠는데. 네. 저희가 얘기에 맞을게 준비하고 있고. 네. 그러다 보니까 나중에 이제 신생아 특례대에서 또 이거 좀 바꿔자 해요. 사실은 다들 그러겠지만. 오제도 오셔가지고 그 말씀해 주시면 되셨어요. 네. 그런데 이제 이 바꿔 나중에 또 혹시 받으려면. 네. 일단 좀 찾아보니까 이게 상한 정도로 되어있으면은. 안 된다고 하는. 상한 용도가 상한 용도로 되어있으면은 이건 그냥 그대로 처음에 구입하셨을 때처럼. 네. 구입 용도로 계속 그걸 지켜드릴 거예요. 아 그렇죠. 네. 그래서 나 좀 걱정 안 하셔도 돼요. 나라 걸로 다시 또 받는다고 하면은. 네. 그렇죠. 그래서 또 차지. 차지. 저한테 물어보세요. 저거 누구한테 물어보세요. 그래도 뭔가 또 가라탄 의미니까. 아니요. 괜찮아요. 그리고 뭐가 문제가 생기면

In [20]:
# audio를 읽어들여 txt파일로 변환 
def transcribe_chunks(chunk_folder, destination):
    files = glob.glob(f"{chunk_folder}/*.mp3")
    model = whisper.load_model("base")
    
    ## 무료버전
    # for file in files:
    #     transcript = model.transcribe(file) # with문으로 as(알리아스) 변수를 사용하면 ndarray 오류가 나므로 파일 바로 사용 
    #     final_transcript += transcript["text"]
            
    # with open(destination, "a") as text_file:
    #     text_file.write(transcript["text"])
    
    ## 유료버전
    for file in files:
        # print(file)
        with open(file, "rb") as audio_file, open(destination, "a") as text_file:
            transcript = openai.Audio.transcribe("whisper-1", audio_file)
            text_file.write(transcript["text"])
        
transcribe_chunks(f"{defaultPath}/files/chunks", f"{defaultPath}/files/transcript.txt")

In [22]:
# audio를 읽어들여 txt파일로 변환 
def transcribe_chunks(chunk_folder, destination):
    files = glob.glob(f"{chunk_folder}/*.mp3")
    model = whisper.load_model("base")
    final_transcript = ""
    
    ## 무료버전 
    for file in files:
        transcript = model.transcribe(file)
        final_transcript += transcript["text"]
            
    with open(destination, "a") as text_file:
        text_file.write(transcript["text"])
    
        
transcribe_chunks(f"{defaultPath}/files/chunks", f"{defaultPath}/files/transcript2.txt")

/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/env/lib/python3.11/site-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/env/lib/python3.11/site-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [34]:
import openai
import json
from pykrx import stock 
from datetime import datetime
import os
import pandas as pd 

openai.api_key = os.getenv("OPENAI_API_KEY")


# 최근 주가 정보 가져오기
def get_stock_price(company_name):
    # 오늘 날짜와 이전 거래일을 구합니다.
    today = datetime.now().strftime("%Y%m%d")
    last_day = stock.get_nearest_business_day_in_a_week(today)

    # 상장된 모든 종목의 정보에서 회사명에 해당하는 종목코드 찾기
    # stock_list = stock.get_market_ticker_name()
    stock_list = pd.DataFrame({'code': stock.get_market_ticker_list(today)})
    stock_list['name'] = stock_list['code'].map(lambda x: stock.get_market_ticker_name(x))

    code = stock_list[stock_list['name']==company_name].code

    # 마지막 거래일의 종가 가져오기
    price_data = stock.get_market_ohlcv_by_date(last_day, last_day, code)

    if not price_data.empty:
        price = str(price_data['종가'].iloc[0])
    else :
        price = "주가정보를 가져올 수 없습니다. 상장된 회사가 맞는지 확인해보세요."

    result = {
        "company_name" : company_name,
        "price" : price,
    }
    
    return json.dumps(result, ensure_ascii=False)

messages = [{"role": "user", "content": "한화시스템의 주가정보 알려줘"}]
functions = [
    {
        "name": "get_stock_price",
        "description": "질문에서 회사명을 찾아 해당 회사의 주가정보를 반환해주는 역할을 하는 함수입니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "한화시스템, 삼성전자, 네이버, 카카오",
                }
            },
            "required": ["company_name"],
        },
    }
]

response = openai.ChatCompletion.create(
    model="gpt-4o-mini",
    messages=messages,
    functions=functions,
    function_call="auto",
    )
response_message = response["choices"][0]["message"]

print(response_message)

if response_message.get("function_call"):
    # Note: the JSON response may not always be valid; be sure to handle errors
    available_functions = {
        "get_stock_price": get_stock_price,
    }
    function_name = response_message["function_call"]["name"]
    fuction_to_call = available_functions[function_name]
    function_args = json.loads(response_message["function_call"]["arguments"])
    function_response = fuction_to_call(
        company_name=function_args.get("company_name"),
    )

    messages.append(response_message)
    messages.append(
        {
            "role": "function",
            "name": function_name,
            "content": function_response,
        }
    )
    second_response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=messages,
    )  # get a new response from GPT where it can see the function response


    json_data = json.dumps(second_response, ensure_ascii=False)

    print(second_response)


{
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_stock_price",
    "arguments": "{\"company_name\":\"\ud55c\ud654\uc2dc\uc2a4\ud15c\"}"
  },
  "refusal": null
}
{
  "id": "chatcmpl-9xWwR42lEyyWtSAFj9blMattn7Dqo",
  "object": "chat.completion",
  "created": 1723975007,
  "model": "gpt-4o-mini-2024-07-18",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "\ud604\uc7ac \ud55c\ud654\uc2dc\uc2a4\ud15c\uc758 \uc8fc\uac00\ub294 19,430\uc6d0\uc785\ub2c8\ub2e4. \ucd94\uac00\uc801\uc778 \uc815\ubcf4\ub098 \ub3c4\uc6c0\uc774 \ud544\uc694\ud558\uc2dc\uba74 \ub9d0\uc500\ud574 \uc8fc\uc138\uc694!",
        "refusal": null
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 62,
    "completion_tokens": 27,
    "total_tokens": 89
  },
  "system_fingerprint": "fp_48196bc67a"
}


In [35]:
second_response

<OpenAIObject chat.completion id=chatcmpl-9xWwR42lEyyWtSAFj9blMattn7Dqo at 0x175e8dc10> JSON: {
  "id": "chatcmpl-9xWwR42lEyyWtSAFj9blMattn7Dqo",
  "object": "chat.completion",
  "created": 1723975007,
  "model": "gpt-4o-mini-2024-07-18",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "\ud604\uc7ac \ud55c\ud654\uc2dc\uc2a4\ud15c\uc758 \uc8fc\uac00\ub294 19,430\uc6d0\uc785\ub2c8\ub2e4. \ucd94\uac00\uc801\uc778 \uc815\ubcf4\ub098 \ub3c4\uc6c0\uc774 \ud544\uc694\ud558\uc2dc\uba74 \ub9d0\uc500\ud574 \uc8fc\uc138\uc694!",
        "refusal": null
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 62,
    "completion_tokens": 27,
    "total_tokens": 89
  },
  "system_fingerprint": "fp_48196bc67a"
}

In [36]:
print(second_response.choices[0].message.content)

현재 한화시스템의 주가는 19,430원입니다. 추가적인 정보나 도움이 필요하시면 말씀해 주세요!
